In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from mtrain.utils import *
from mtrain.example_dir.core import load_npz
from mtrain.utils import overlay_mask_on_img as OV

what are my goals?

- find as many false positives as possible
    - find larger false positives (the most obvious ones)
- find stuff model might have missed using big differences with high recall model

the meaning of large can be different:
- total number of garbage pixels
- total number of regions
- size of single regions (finding big regions as an example)



first:
- find obvious false positives

In [ ]:
chunks_dir = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi/chunks")

In [ ]:
all_dirs = []

for name in list(map(str, range(35))):
    chunk_dir = chunks_dir / name
    for d in chunk_dir.glob("*"):
        ntp, nop = d / "negmask-trash-md.npz", d / "negmask-other-md.npz"
        if ntp.exists() and nop.exists():
            all_dirs.append(d)

In [ ]:
dirs = random.sample(all_dirs, 5000)

In [ ]:
! ls {dirs[0]}

In [ ]:
idx = 4

In [ ]:
from tqdm import tqdm

def trash_mask(d, label):
    t = load_npz(d / f"negmask-trash-{label}.npz")
    o = load_npz(d / f"negmask-other-{label}.npz")
    return t > o


def get_high_trash_pixel_counts(dirs):
    counts = []
    for d in tqdm(dirs, desc="High trash pixel count"):
        mask = DiskBooleanMask.load(d / "mask-md.png")
        tmask = trash_mask(d, "md")
        final = mask & tmask
        counts.append((final.sum(), d))
    counts = list(reversed(sorted(counts)))
    return counts

In [ ]:
hp_dirs = get_high_trash_pixel_counts(dirs[:1000])

Problems:
- Painted walls
- Railings/fences
- rock like walls
- sunlights
- blurry images (these need to be handled for sure)
- segmask model overflows a bit outside the actual object
- railway tracks
- smallnet has issues with collections of trash (this is one of the bigger painpoints, i think we can train it with the additional data from delhi)
- puddles of water
- rain water on car glass / lens glass
- hawkers

Now it seems its best to assign new labels to these and train the model on them (we do have a lot of unlabelled data for that case though). A new backbone can be trained which does that, then we trim the older data, then train the older backbone again. 


The easiest way to get more data is to simply get images where the model has mis predicted everything and add them. I now need faster data gathering stuff though. It is useful to train another model which has high other recall and check the differences too (the other way round)

In [ ]:
from mtrain.seg.mapillary import show_seg_mask, get_mask_with_labels, Label as L


d = hp_dirs[9][1]
img = DiskImage.load(d / "image.jpg")
mapi = DiskBooleanMask.load(d / "mapi.png")
mask = DiskBooleanMask.load(d / "mask-md.png")
tmask = trash_mask(d, "md")

with_labels = get_mask_with_labels(mapi, [L.WALL, L.FENCE, L.BRIDGE])
mask_with_labels = with_labels & tmask

final = mask & tmask

show([img, OV(img, final), OV(img, mask_with_labels), OV(img, mask)], (20,20), 2, "off")

In [ ]:

show_seg_mask(mapi)